In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import MatchEnv, PoolController, RandomController
from src.rl.env_wrapper import Stage2OpponentController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomReset
from src.rl.reward_shapers import DenseReward_2
from src.rl.trainer import train_ppo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

In [ ]:
# ── CONFIGURATION ──
STAGE = 3  # 1: vs Random, 2: vs Heuristic, 3: vs Mixed Pool
LEARNER_TEAM = "red"
OPPONENT_TEAM = "blue"

SAVE_DIR = f"models/stage{STAGE}"
POOL_DIR = f"models/stage{STAGE}/pool" if STAGE == 3 else None
os.makedirs(SAVE_DIR, exist_ok=True)


def make_env():
    roster = [
        PlayerSlot(LEARNER_TEAM, PlayerStats(name="Learner", accel=3200.0), controller="RL"),
    ]

    if STAGE == 1:
        opp_ctrl = RandomController()
    elif STAGE == 2:
        opp_ctrl = HeuristicBotController(TeamHeuristicCoordinator(OPPONENT_TEAM))
    else:
        opp_ctrl = PoolController(pool_dir=POOL_DIR, device="cpu")

    roster.append(PlayerSlot(OPPONENT_TEAM, PlayerStats(name="Opponent", accel=3200.0), controller=opp_ctrl))

    cfg = MatchConfig(mode=ClassicMatchMode(time_limit=30.0, score_limit=99), roster=roster)

    return MatchEnv(
        match_config=cfg,
        reward_shaper=DenseReward_2(team=LEARNER_TEAM),
        reset_strategy=RandomReset(),
        learner_team=LEARNER_TEAM,
        max_steps=1800,
    )


NUM_ENVS = 16
train_envs = gym.vector.SyncVectorEnv([make_env for _ in range(NUM_ENVS)])
model = ActorCritic(obs_dim=80).to(device)

if STAGE > 1:
    prev_ckpt = f"models/stage{STAGE-1}/best_model.pt"
    if os.path.exists(prev_ckpt):
        model.load_state_dict(torch.load(prev_ckpt, map_location=device, weights_only=False))
        print(f"✅ Loaded checkpoint from Stage {STAGE-1}")

train_ppo(
    envs=train_envs,
    model=model,
    device=device,
    max_steps=1800,
    time_limit=30.0,
    baseline_type="random" if STAGE == 1 else "heuristic",
    total_timesteps=15_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
    lr_initial=3e-4 if STAGE == 1 else 5e-5,
    lr_final=1e-5,
    ent_coef_initial=0.015 if STAGE == 1 else 0.005,
    ent_coef_final=0.0005,
    
)

train_envs.close()

In [ ]:
import os
import sys
import torch

sys.path.insert(0, os.path.abspath(".."))
from src.rl.evaluator import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Render 5 evaluation matches of your best checkpoint
replay_file = evaluate_and_generate_html(
    model_or_path="models/stage3/best_model.pt",
    device=device,
    baseline_type="heuristic",  # Visualizes matches against Heuristic bot
    output_dir="render/",
    filename="stage3_diagnostic.html",
    num_episodes=5,
    max_steps=1800,  # 30 seconds per match
)